In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", None)

In [2]:
METADATA_CSV = Path("/kaggle/input/datasets/khanhngnguynl/deepfashion/results/data/processed/embeddings/metadata.csv")
LIST_EVAL_PARTITION_TXT = Path("/kaggle/input/datasets/khanhngnguynl/deepfashion/list_eval_partition.txt")

ORIGINAL_EMBEDDINGS_NPY = Path("/kaggle/input/datasets/khanhngnguynl/deepfashion/results/data/processed/embeddings/original_embeddings.npy")
CROPPED_EMBEDDINGS_NPY = Path("/kaggle/input/datasets/khanhngnguynl/deepfashion/results/data/processed/embeddings/cropped_embeddings.npy")

OUTPUT_DIR = Path("/kaggle/working/ground_truth")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_ITEM_OVERLAP_RATIO = 0.5
MIN_EXACT_IMAGE_MATCH_RATIO = 0.9

print("METADATA_CSV            :", METADATA_CSV, "| exists:", METADATA_CSV.exists())
print("LIST_EVAL_PARTITION_TXT :", LIST_EVAL_PARTITION_TXT, "| exists:", LIST_EVAL_PARTITION_TXT.exists())
print("ORIGINAL_EMBEDDINGS_NPY :", ORIGINAL_EMBEDDINGS_NPY, "| exists:", ORIGINAL_EMBEDDINGS_NPY.exists())
print("CROPPED_EMBEDDINGS_NPY  :", CROPPED_EMBEDDINGS_NPY, "| exists:", CROPPED_EMBEDDINGS_NPY.exists())
print("OUTPUT_DIR               :", OUTPUT_DIR)

METADATA_CSV            : /kaggle/input/datasets/khanhngnguynl/deepfashion/results/data/processed/embeddings/metadata.csv | exists: True
LIST_EVAL_PARTITION_TXT : /kaggle/input/datasets/khanhngnguynl/deepfashion/list_eval_partition.txt | exists: True
ORIGINAL_EMBEDDINGS_NPY : /kaggle/input/datasets/khanhngnguynl/deepfashion/results/data/processed/embeddings/original_embeddings.npy | exists: True
CROPPED_EMBEDDINGS_NPY  : /kaggle/input/datasets/khanhngnguynl/deepfashion/results/data/processed/embeddings/cropped_embeddings.npy | exists: True
OUTPUT_DIR               : /kaggle/working/ground_truth


In [3]:
def load_metadata(metadata_csv):
    if not metadata_csv.exists():
        raise FileNotFoundError(
            f"Không tìm thấy metadata.csv tại: {metadata_csv}"
        )
    df = pd.read_csv(metadata_csv)
    required_cols = {"image_id", "image_path"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(
            f"metadata.csv thiếu cột bắt buộc: {missing_cols}"
        )
    before = len(df)
    df = (
        df.dropna(subset=["image_id", "image_path"])
          .drop_duplicates(subset=["image_id"])
    )
    after = len(df)
    if after < before:
        print(
            f"[WARNING] metadata.csv có "
            f"{before - after} dòng bị loại do NaN hoặc duplicate image_id."
        )
    OLD_ROOT = Path("/kaggle/input/datasets/khanhngnguynl/deepfashion/images")
    NEW_ROOT = Path("/kaggle/input/datasets/khanhngnguynl/deepfashion/deepfashion/images")

    def update_image_path(path):
        path = Path(str(path))

        try:
            relative_path = path.relative_to(OLD_ROOT)
            return NEW_ROOT / relative_path
        except ValueError:
            return path

    df["image_path_updated"] = df["image_path"].apply(update_image_path)

    df["old_path_exists"] = df["image_path"].apply(
        lambda x: Path(str(x)).exists()
    )

    df["new_path_exists"] = df["image_path_updated"].apply(
        lambda x: Path(x).exists()
    )

    old_exists = df["old_path_exists"].sum()
    new_exists = df["new_path_exists"].sum()
    missing = (~df["new_path_exists"]).sum()

    print()
    print("=" * 60)
    print("IMAGE PATH VALIDATION")
    print("=" * 60)
    print(f"Total images          : {len(df)}")
    print(f"Old path exists       : {old_exists}")
    print(f"New path exists       : {new_exists}")
    print(f"Missing new paths     : {missing}")
    print("=" * 60)

    if missing > 0:
        print(
            f"[WARNING] Có {missing} image không tồn tại "
            f"ở dataset path mới."
        )

        print("\nMột số missing paths:")
        print(
            df.loc[
                ~df["new_path_exists"],
                ["image_id", "image_path_updated"]
            ].head(10).to_string(index=False)
        )

    return df.reset_index(drop=True)


def load_inshop_partition(path):
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy list_eval_partition.txt tại: {path}")

    try:
        with open(path, "r", encoding="utf-8") as f:
            lines = f.readlines()
    except UnicodeDecodeError:
        with open(path, "r", encoding="latin-1") as f:
            lines = f.readlines()

    # Format chuẩn DeepFashion In-shop:
    # line 0: số lượng ảnh
    # line 1: header "image_name item_id evaluation_status"
    # line 2..: dữ liệu, phân tách bởi khoảng trắng
    if len(lines) < 2:
        raise ValueError("File list_eval_partition.txt có định dạng không hợp lệ (quá ít dòng).")

    header_line = lines[1].strip().split()
    data_lines = lines[2:]

    records = []
    for line in data_lines:
        parts = line.strip().split()
        if len(parts) != 3:
            continue
        records.append(parts)

    df = pd.DataFrame(records, columns=header_line if len(header_line) == 3
                       else ["image_name", "item_id", "evaluation_status"])

    before = len(df)
    df = df.dropna().drop_duplicates()
    after = len(df)
    if after < before:
        print(f"[WARNING] list_eval_partition.txt có {before - after} dòng bị loại do NaN/duplicate.")

    return df.reset_index(drop=True)


def load_embeddings_for_compat_check(original_path, cropped_path):
    result = {}
    for name, path in [("original", original_path), ("cropped", cropped_path)]:
        if path.exists():
            result[name] = np.load(path)
        else:
            print(f"[WARNING] Không tìm thấy {path} - bỏ qua kiểm tra compatibility cho '{name}' embedding.")
            result[name] = None
    return result


dfm_df = load_metadata(METADATA_CSV)
inshop_df = load_inshop_partition(LIST_EVAL_PARTITION_TXT)
embeddings_dict = load_embeddings_for_compat_check(ORIGINAL_EMBEDDINGS_NPY, CROPPED_EMBEDDINGS_NPY)

print("DFM metadata rows   :", len(dfm_df))
print("In-shop rows        :", len(inshop_df))
dfm_df.head()


IMAGE PATH VALIDATION
Total images          : 12701
Old path exists       : 0
New path exists       : 12701
Missing new paths     : 0
DFM metadata rows   : 12701
In-shop rows        : 52712


,image_id,image_path,image_path_updated,old_path_exists,new_path_exists
0,MEN-Denim-id_00000080-01_7_additional,/kaggle/input/datasets/khanhngnguynl/deepfashion/images/MEN-Denim-id_00000080-01_7_additional.jpg,/kaggle/input/datasets/khanhngnguynl/deepfashion/deepfashion/images/MEN-Denim-id_00000080-01_7_additional.jpg,False,True
1,MEN-Denim-id_00000089-01_7_additional,/kaggle/input/datasets/khanhngnguynl/deepfashion/images/MEN-Denim-id_00000089-01_7_additional.jpg,/kaggle/input/datasets/khanhngnguynl/deepfashion/deepfashion/images/MEN-Denim-id_00000089-01_7_additional.jpg,False,True
2,MEN-Denim-id_00000089-02_7_additional,/kaggle/input/datasets/khanhngnguynl/deepfashion/images/MEN-Denim-id_00000089-02_7_additional.jpg,/kaggle/input/datasets/khanhngnguynl/deepfashion/deepfashion/images/MEN-Denim-id_00000089-02_7_additional.jpg,False,True
3,MEN-Denim-id_00000089-03_7_additional,/kaggle/input/datasets/khanhngnguynl/deepfashion/images/MEN-Denim-id_00000089-03_7_additional.jpg,/kaggle/input/datasets/khanhngnguynl/deepfashion/deepfashion/images/MEN-Denim-id_00000089-03_7_additional.jpg,False,True
4,MEN-Denim-id_00000089-04_7_additional,/kaggle/input/datasets/khanhngnguynl/deepfashion/images/MEN-Denim-id_00000089-04_7_additional.jpg,/kaggle/input/datasets/khanhngnguynl/deepfashion/deepfashion/images/MEN-Denim-id_00000089-04_7_additional.jpg,False,True


In [4]:
def check_embedding_compatibility(dfm_df, embeddings_dict):
    print("=" * 50)
    print("PHẦN 6 - KIỂM TRA TƯƠNG THÍCH VỚI EMBEDDING")
    print("=" * 50)
    print()

    n_metadata = len(dfm_df)
    print(f"len(metadata) = {n_metadata}")

    all_ok = True
    for name, emb in embeddings_dict.items():
        if emb is None:
            print(f"len({name}_embeddings) = KHÔNG TÌM THẤY FILE, bỏ qua kiểm tra")
            continue
        n_emb = len(emb)
        status = "OK" if n_emb == n_metadata else "MISMATCH"
        if status == "MISMATCH":
            all_ok = False
        print(f"len({name}_embeddings) = {n_emb}  ->  {status}")

    print()
    if not all_ok:
        print("[ERROR] Phát hiện mismatch giữa metadata.csv và embedding .npy. "
              "KHÔNG được tiếp tục sử dụng embedding cho evaluation cho đến khi khắc phục. "
              "Không tự động reorder hoặc sửa embedding.")
    else:
        print("Số lượng metadata và các embedding khớp nhau. "
              "Lưu ý: script này chỉ kiểm tra SỐ LƯỢNG, không thể tự xác nhận "
              "thứ tự từng dòng là chính xác nếu file gốc từng bị sắp xếp lại thủ công.")
    print()
    print("=" * 50)

    return all_ok


embedding_compat_ok = check_embedding_compatibility(dfm_df, embeddings_dict)

PHẦN 6 - KIỂM TRA TƯƠNG THÍCH VỚI EMBEDDING

len(metadata) = 12701
len(original_embeddings) = 12701  ->  OK
len(cropped_embeddings) = 12701  ->  OK

Số lượng metadata và các embedding khớp nhau. Lưu ý: script này chỉ kiểm tra SỐ LƯỢNG, không thể tự xác nhận thứ tự từng dòng là chính xác nếu file gốc từng bị sắp xếp lại thủ công.



In [5]:
def extract_dfm_item_id(image_id):
    """VD: MEN-Denim-id_00000089-01_7_additional -> id_00000089"""
    match = re.search(r"(id_\d+)", image_id)
    return match.group(1) if match else None


def extract_dfm_suffix(image_id):
    """
    Lấy phần suffix sau item_id, ví dụ:
    MEN-Denim-id_00000089-01_7_additional -> "01_7_additional.jpg"
    Trả về None nếu không tách được (không giả định luôn tồn tại).
    """
    match = re.search(r"id_\d+-(.+)$", image_id)
    if not match:
        return None
    return match.group(1).lower() + ".jpg"


def normalize_inshop_image_name(image_name):
    """Lấy basename (tên file) của path In-shop, lowercase, để so khớp suffix DFM."""
    return Path(image_name).name.lower()


def build_mapping_table(dfm_df, inshop_df):
    """
    mapping_level:
        exact_image -> tìm được đúng (item_id, suffix_filename) trong In-shop
        item_only   -> item_id có trong In-shop nhưng không tìm được đúng file
        unmatched   -> item_id không có trong In-shop

    mapping_confidence:
        high   <- exact_image
        medium <- item_only
        low    <- unmatched
    """
    inshop_df = inshop_df.copy()
    inshop_df["inshop_suffix"] = inshop_df["image_name"].apply(normalize_inshop_image_name)

    key_counts = inshop_df.groupby(["item_id", "inshop_suffix"]).size()
    duplicate_keys = key_counts[key_counts > 1]
    n_duplicate_keys = len(duplicate_keys)
    if n_duplicate_keys > 0:
        print(f"[WARNING] Có {n_duplicate_keys} cặp (item_id, filename) bị trùng trong In-shop "
              f"-> khi match sẽ lấy dòng đầu tiên, có thể gây ambiguity nhẹ.")

    inshop_lookup = {}
    for _, row in inshop_df.iterrows():
        key = (row["item_id"], row["inshop_suffix"])
        if key not in inshop_lookup:
            inshop_lookup[key] = (row["image_name"], row["evaluation_status"])

    inshop_item_ids = set(inshop_df["item_id"].unique())

    records = []
    for _, row in dfm_df.iterrows():
        dfm_image_id = row["image_id"]
        dfm_image_path = row["image_path"]

        dfm_item_id = extract_dfm_item_id(dfm_image_id)
        dfm_suffix = extract_dfm_suffix(dfm_image_id)

        if dfm_item_id is None:
            records.append({
                "dfm_image_id": dfm_image_id, "dfm_image_path": dfm_image_path,
                "dfm_item_id": None, "inshop_item_id": None, "inshop_image_name": None,
                "evaluation_status": None, "mapping_level": "unmatched", "mapping_confidence": "low",
            })
            continue

        key = (dfm_item_id, dfm_suffix) if dfm_suffix else None
        exact_match = inshop_lookup.get(key) if key else None

        if exact_match is not None:
            inshop_image_name, evaluation_status = exact_match
            records.append({
                "dfm_image_id": dfm_image_id, "dfm_image_path": dfm_image_path,
                "dfm_item_id": dfm_item_id, "inshop_item_id": dfm_item_id,
                "inshop_image_name": inshop_image_name, "evaluation_status": evaluation_status,
                "mapping_level": "exact_image", "mapping_confidence": "high",
            })
        elif dfm_item_id in inshop_item_ids:
            records.append({
                "dfm_image_id": dfm_image_id, "dfm_image_path": dfm_image_path,
                "dfm_item_id": dfm_item_id, "inshop_item_id": dfm_item_id,
                "inshop_image_name": None, "evaluation_status": None,
                "mapping_level": "item_only", "mapping_confidence": "medium",
            })
        else:
            records.append({
                "dfm_image_id": dfm_image_id, "dfm_image_path": dfm_image_path,
                "dfm_item_id": dfm_item_id, "inshop_item_id": None,
                "inshop_image_name": None, "evaluation_status": None,
                "mapping_level": "unmatched", "mapping_confidence": "low",
            })

    mapping_df = pd.DataFrame(records)
    return mapping_df, n_duplicate_keys


mapping_df, n_duplicate_keys = build_mapping_table(dfm_df, inshop_df)
mapping_df.head()

[WARNING] Có 37 cặp (item_id, filename) bị trùng trong In-shop -> khi match sẽ lấy dòng đầu tiên, có thể gây ambiguity nhẹ.


,dfm_image_id,dfm_image_path,dfm_item_id,inshop_item_id,inshop_image_name,evaluation_status,mapping_level,mapping_confidence
0,MEN-Denim-id_00000080-01_7_additional,/kaggle/input/datasets/khanhngnguynl/deepfashion/images/MEN-Denim-id_00000080-01_7_additional.jpg,id_00000080,id_00000080,img/MEN/Denim/id_00000080/01_7_additional.jpg,train,exact_image,high
1,MEN-Denim-id_00000089-01_7_additional,/kaggle/input/datasets/khanhngnguynl/deepfashion/images/MEN-Denim-id_00000089-01_7_additional.jpg,id_00000089,id_00000089,img/MEN/Denim/id_00000089/01_7_additional.jpg,train,exact_image,high
2,MEN-Denim-id_00000089-02_7_additional,/kaggle/input/datasets/khanhngnguynl/deepfashion/images/MEN-Denim-id_00000089-02_7_additional.jpg,id_00000089,id_00000089,img/MEN/Denim/id_00000089/02_7_additional.jpg,train,exact_image,high
3,MEN-Denim-id_00000089-03_7_additional,/kaggle/input/datasets/khanhngnguynl/deepfashion/images/MEN-Denim-id_00000089-03_7_additional.jpg,id_00000089,id_00000089,img/MEN/Denim/id_00000089/03_7_additional.jpg,train,exact_image,high
4,MEN-Denim-id_00000089-04_7_additional,/kaggle/input/datasets/khanhngnguynl/deepfashion/images/MEN-Denim-id_00000089-04_7_additional.jpg,id_00000089,id_00000089,img/MEN/Denim/id_00000089/04_7_additional.jpg,train,exact_image,high


In [6]:
def compute_overview_statistics(dfm_df, inshop_df, mapping_df):
    dfm_total_images = len(dfm_df)
    dfm_unique_items = mapping_df["dfm_item_id"].nunique(dropna=True)

    inshop_total_images = len(inshop_df)
    inshop_unique_items = inshop_df["item_id"].nunique()

    matched_items = mapping_df.loc[mapping_df["mapping_level"] != "unmatched", "dfm_item_id"].nunique()
    unmatched_items = dfm_unique_items - matched_items

    dfm_images_matched_item = (mapping_df["mapping_level"] != "unmatched").sum()
    dfm_images_unmatched = (mapping_df["mapping_level"] == "unmatched").sum()

    dfm_images_exact = (mapping_df["mapping_level"] == "exact_image").sum()
    dfm_images_item_only = (mapping_df["mapping_level"] == "item_only").sum()

    item_overlap_ratio = matched_items / dfm_unique_items if dfm_unique_items > 0 else 0.0
    exact_image_ratio = dfm_images_exact / dfm_total_images if dfm_total_images > 0 else 0.0

    return {
        "dfm_total_images": dfm_total_images, "dfm_unique_items": dfm_unique_items,
        "inshop_total_images": inshop_total_images, "inshop_unique_items": inshop_unique_items,
        "matched_items": matched_items, "unmatched_items": unmatched_items,
        "item_overlap_ratio": round(item_overlap_ratio, 4),
        "dfm_images_matched_item": int(dfm_images_matched_item),
        "dfm_images_unmatched": int(dfm_images_unmatched),
        "dfm_images_exact_image": int(dfm_images_exact),
        "dfm_images_item_only": int(dfm_images_item_only),
        "exact_image_match_ratio": round(exact_image_ratio, 4),
    }


def print_overview_statistics(stats):
    print("=" * 50)
    print("DFM <-> IN-SHOP MAPPING CHECK")
    print("=" * 50)
    print()
    print(f"DFM images              : {stats['dfm_total_images']}")
    print(f"DFM unique item IDs     : {stats['dfm_unique_items']}")
    print()
    print(f"In-shop images          : {stats['inshop_total_images']}")
    print(f"In-shop unique item IDs : {stats['inshop_unique_items']}")
    print()
    print(f"Matched item IDs        : {stats['matched_items']}")
    print(f"Unmatched item IDs      : {stats['unmatched_items']}")
    print(f"Item overlap ratio      : {stats['item_overlap_ratio']:.2%}")
    print()
    print(f"DFM images with matched item ID : {stats['dfm_images_matched_item']}")
    print(f"DFM images unmatched             : {stats['dfm_images_unmatched']}")
    print()
    print(f"DFM images - exact_image match   : {stats['dfm_images_exact_image']}")
    print(f"DFM images - item_only match     : {stats['dfm_images_item_only']}")
    print(f"Exact image match ratio (of all DFM images): {stats['exact_image_match_ratio']:.2%}")
    print()
    print("=" * 50)


overview_stats = compute_overview_statistics(dfm_df, inshop_df, mapping_df)
print_overview_statistics(overview_stats)

DFM <-> IN-SHOP MAPPING CHECK

DFM images              : 12701
DFM unique item IDs     : 6902

In-shop images          : 52712
In-shop unique item IDs : 7982

Matched item IDs        : 6902
Unmatched item IDs      : 0
Item overlap ratio      : 100.00%

DFM images with matched item ID : 12701
DFM images unmatched             : 0

DFM images - exact_image match   : 12701
DFM images - item_only match     : 0
Exact image match ratio (of all DFM images): 100.00%



In [7]:
mapping_result_path = OUTPUT_DIR / "mapping_result.csv"
mapping_df.to_csv(mapping_result_path, index=False)
print(f"Đã lưu: {mapping_result_path}")

mapping_statistics_path = OUTPUT_DIR / "mapping_statistics.csv"
stats_df = pd.DataFrame([{**overview_stats, "duplicate_inshop_keys": n_duplicate_keys}])
stats_df.to_csv(mapping_statistics_path, index=False)
print(f"Đã lưu: {mapping_statistics_path}")

stats_df

Đã lưu: /kaggle/working/ground_truth/mapping_result.csv
Đã lưu: /kaggle/working/ground_truth/mapping_statistics.csv


,dfm_total_images,dfm_unique_items,inshop_total_images,inshop_unique_items,matched_items,unmatched_items,item_overlap_ratio,dfm_images_matched_item,dfm_images_unmatched,dfm_images_exact_image,dfm_images_item_only,exact_image_match_ratio,duplicate_inshop_keys
0,12701,6902,52712,7982,6902,0,1.0,12701,0,12701,0,1.0,37


In [8]:
def analyze_query_gallery(mapping_df):
    """Chỉ xét exact_image, vì chỉ những dòng này mới biết chắc evaluation_status."""
    exact_df = mapping_df[mapping_df["mapping_level"] == "exact_image"]

    n_query_images = (exact_df["evaluation_status"] == "query").sum()
    n_gallery_images = (exact_df["evaluation_status"] == "gallery").sum()
    n_train_images = (exact_df["evaluation_status"] == "train").sum()
    n_unassigned_images = len(mapping_df) - len(exact_df)

    query_items = set(exact_df.loc[exact_df["evaluation_status"] == "query", "dfm_item_id"])
    gallery_items = set(exact_df.loc[exact_df["evaluation_status"] == "gallery", "dfm_item_id"])
    items_in_both = query_items & gallery_items

    return {
        "dfm_query_images": int(n_query_images),
        "dfm_gallery_images": int(n_gallery_images),
        "dfm_train_images": int(n_train_images),
        "dfm_unassigned_images": int(n_unassigned_images),
        "query_items": len(query_items),
        "gallery_items": len(gallery_items),
        "items_in_both_query_and_gallery": len(items_in_both),
    }


def print_query_gallery_stats(stats):
    print("=" * 50)
    print("QUERY / GALLERY CHECK (chỉ trên exact_image matches)")
    print("=" * 50)
    print()
    print(f"DFM query images    : {stats['dfm_query_images']}")
    print(f"DFM gallery images  : {stats['dfm_gallery_images']}")
    print(f"DFM train images    : {stats['dfm_train_images']}")
    print(f"DFM unassigned      : {stats['dfm_unassigned_images']}")
    print()
    print(f"Query items         : {stats['query_items']}")
    print(f"Gallery items       : {stats['gallery_items']}")
    print(f"Items in BOTH query & gallery : {stats['items_in_both_query_and_gallery']}")
    print()
    print("=" * 50)


qg_stats = analyze_query_gallery(mapping_df)
print_query_gallery_stats(qg_stats)

QUERY / GALLERY CHECK (chỉ trên exact_image matches)

DFM query images    : 3540
DFM gallery images  : 2944
DFM train images    : 6217
DFM unassigned      : 0

Query items         : 2444
Gallery items       : 2045
Items in BOTH query & gallery : 1042



In [9]:
def decide_evaluation_validity(overview_stats):
    item_overlap_ratio = overview_stats["item_overlap_ratio"]
    exact_image_ratio = overview_stats["exact_image_match_ratio"]

    if item_overlap_ratio < MIN_ITEM_OVERLAP_RATIO:
        case = "C"
        message = "NOT SUITABLE FOR DIRECT EVALUATION"
        reason = (f"Item overlap ratio ({item_overlap_ratio:.2%}) thấp hơn ngưỡng "
                  f"MIN_ITEM_OVERLAP_RATIO ({MIN_ITEM_OVERLAP_RATIO:.2%}).")
    elif exact_image_ratio >= MIN_EXACT_IMAGE_MATCH_RATIO:
        case = "A"
        message = "VALID FOR IMAGE-LEVEL OFFICIAL EVALUATION"
        reason = (f"Exact image match ratio ({exact_image_ratio:.2%}) đạt ngưỡng "
                  f"MIN_EXACT_IMAGE_MATCH_RATIO ({MIN_EXACT_IMAGE_MATCH_RATIO:.2%}).")
    else:
        case = "B"
        message = "VALID ONLY FOR ITEM-LEVEL GROUND TRUTH, IMAGE-LEVEL PARTITION TRANSFER NOT PROVEN"
        reason = (f"Item overlap ratio ({item_overlap_ratio:.2%}) đủ để dùng ở mức item, "
                  f"nhưng exact image match ratio ({exact_image_ratio:.2%}) chưa đạt ngưỡng "
                  f"MIN_EXACT_IMAGE_MATCH_RATIO ({MIN_EXACT_IMAGE_MATCH_RATIO:.2%}).")

    return case, message, reason


case, message, reason = decide_evaluation_validity(overview_stats)

print("=" * 50)
print("PHẦN 4 - QUYẾT ĐỊNH CUỐI CÙNG")
print("=" * 50)
print()
print(f"Case: {case}")
print(f"Kết luận: {message}")
print(f"Lý do: {reason}")
print()
print("=" * 50)

PHẦN 4 - QUYẾT ĐỊNH CUỐI CÙNG

Case: A
Kết luận: VALID FOR IMAGE-LEVEL OFFICIAL EVALUATION
Lý do: Exact image match ratio (100.00%) đạt ngưỡng MIN_EXACT_IMAGE_MATCH_RATIO (90.00%).



In [10]:
import pandas as pd

def build_ground_truth(mapping_df):
    exact_df = mapping_df[mapping_df["mapping_level"] == "exact_image"].copy()

    if len(exact_df) == 0:
        raise ValueError("Không có mapping_level == 'exact_image'. Không thể xây dựng Ground Truth.")

    query_all_df = exact_df[exact_df["evaluation_status"] == "query"].reset_index(drop=True)
    gallery_df = exact_df[exact_df["evaluation_status"] == "gallery"].reset_index(drop=True)

    query_all_df = query_all_df.reset_index().rename(columns={"index": "query_index"})
    gallery_df = gallery_df.reset_index().rename(columns={"index": "gallery_index"})

    gallery_by_item = {item_id: group for item_id, group in gallery_df.groupby("dfm_item_id")}

    valid_query_indices = []
    invalid_query_indices = []

    for _, q_row in query_all_df.iterrows():
        item_id = q_row["dfm_item_id"]
        if item_id in gallery_by_item:
            valid_query_indices.append(q_row["query_index"])
        else:
            invalid_query_indices.append(q_row["query_index"])


    query_df = query_all_df[query_all_df["query_index"].isin(valid_query_indices)].reset_index(drop=True)
    queries_without_positive = query_all_df[query_all_df["query_index"].isin(invalid_query_indices)].copy()

    gt_records = []
    for _, q_row in query_df.iterrows():
        item_id = q_row["dfm_item_id"]
        for _, g_row in gallery_by_item[item_id].iterrows():
            gt_records.append({
                "query_index": q_row["query_index"],
                "query_image_id": q_row["dfm_image_id"],
                "query_item_id": item_id,
                "gallery_index": g_row["gallery_index"],
                "positive_gallery_image_id": g_row["dfm_image_id"],
                "positive_gallery_item_id": g_row["dfm_item_id"],
            })

    ground_truth_df = pd.DataFrame(gt_records)

    print(f"\n{'=' * 60}\nGROUND TRUTH SUMMARY\n{'=' * 60}")
    print(f"Original query images    : {len(query_all_df)}")
    print(f"Valid query images       : {len(query_df)}")
    print(f"Queries without positive : {len(queries_without_positive)}")
    print(f"Gallery images           : {len(gallery_df)}")
    print(f"Query-positive pairs     : {len(ground_truth_df)}")

    if len(query_all_df) > 0:
        coverage = (len(query_df) / len(query_all_df)) * 100
        print(f"Valid query coverage     : {coverage:.2f}%")
    print("=" * 60)

    return query_df, gallery_df, ground_truth_df, queries_without_positive


if case == "A":
    print("Mapping đủ điều kiện Case A -> tiến hành xây dựng Ground Truth.")

    query_df, gallery_df, ground_truth_df, queries_without_positive = build_ground_truth(mapping_df)

    # OUTPUT PATHS & SAVE FILES
    query_path = OUTPUT_DIR / "query.csv"
    gallery_path = OUTPUT_DIR / "gallery.csv"
    gt_path = OUTPUT_DIR / "ground_truth.csv"
    removed_query_path = OUTPUT_DIR / "query_without_positive.csv"

    query_df.to_csv(query_path, index=False, encoding="utf-8-sig")
    gallery_df.to_csv(gallery_path, index=False, encoding="utf-8-sig")
    ground_truth_df.to_csv(gt_path, index=False, encoding="utf-8-sig")
    queries_without_positive.to_csv(removed_query_path, index=False, encoding="utf-8-sig")

    # FINAL OUTPUT
    print(f"\n{'=' * 60}\nGROUND TRUTH FILES\n{'=' * 60}")
    print(f"Đã lưu: {query_path} ({len(query_df)} query images)")
    print(f"Đã lưu: {gallery_path} ({len(gallery_df)} gallery images)")
    print(f"Đã lưu: {gt_path} ({len(ground_truth_df)} query-positive pairs)")
    print(f"Đã lưu: {removed_query_path} ({len(queries_without_positive)} removed queries)")
    print("=" * 60)

    # FINAL VALIDATION
    if len(queries_without_positive) > 0:
        print(f"\n[INFO] {len(queries_without_positive)} query không có positive gallery đã được loại khỏi query.csv.")
        print(f"[INFO] Các query này vẫn được lưu tại:\n       {removed_query_path}")

    # Kiểm tra tất cả query trong query.csv đều có GT
    missing_gt = set(query_df["dfm_image_id"]) - set(ground_truth_df["query_image_id"])
    if missing_gt:
        print(f"[ERROR] Có {len(missing_gt)} query trong query.csv nhưng không có Ground Truth.")
    else:
        print("[OK] Tất cả query trong query.csv đều có ít nhất một positive gallery.")

else:
    print(f"Không tạo query.csv / gallery.csv / ground_truth.csv vì kết quả là Case {case}.")
    print("Chỉ có mapping_result.csv và mapping_statistics.csv được tạo.")
    print(f"Lý do: {reason}")

Mapping đủ điều kiện Case A -> tiến hành xây dựng Ground Truth.

GROUND TRUTH SUMMARY
Original query images    : 3540
Valid query images       : 1890
Queries without positive : 1650
Gallery images           : 2944
Query-positive pairs     : 4697
Valid query coverage     : 53.39%

GROUND TRUTH FILES
Đã lưu: /kaggle/working/ground_truth/query.csv (1890 query images)
Đã lưu: /kaggle/working/ground_truth/gallery.csv (2944 gallery images)
Đã lưu: /kaggle/working/ground_truth/ground_truth.csv (4697 query-positive pairs)
Đã lưu: /kaggle/working/ground_truth/query_without_positive.csv (1650 removed queries)

[INFO] 1650 query không có positive gallery đã được loại khỏi query.csv.
[INFO] Các query này vẫn được lưu tại:
       /kaggle/working/ground_truth/query_without_positive.csv
[OK] Tất cả query trong query.csv đều có ít nhất một positive gallery.
